# 🏥 Medical Triage System — Training & Testing

This notebook trains and evaluates **two models** on the Kaggle Medical Transcriptions dataset:
1. **LSTM** (fast baseline) — TensorFlow/Keras
2. **BioBERT** (advanced) — PyTorch/HuggingFace

> **Environment:** Designed to run directly on **Kaggle Notebooks** with GPU enabled.
> Add the dataset: [Medical Transcriptions](https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions)

## 1. Setup & Installation

In [ ]:
!pip install -q transformers tqdm

## 2. Imports

In [ ]:
import os, re, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings('ignore')
print('All imports successful.')
print(f'GPU available: {torch.cuda.is_available()}')

## 3. Load Dataset from Kaggle Input

In [ ]:
# Kaggle mounts added datasets under /kaggle/input/<dataset-slug>/
KAGGLE_PATH = '/kaggle/input/medicaltranscriptions/mtsamples.csv'
LOCAL_PATH  = 'data/mtsamples.csv'

data_path = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else LOCAL_PATH
print(f'Loading dataset from: {data_path}')

df = pd.read_csv(data_path)
print(f'Raw dataset shape: {df.shape}')
df.head()

## 4. Exploratory Data Analysis

In [ ]:
df = df.dropna(subset=['transcription', 'medical_specialty'])
print(f'After dropping NaN: {df.shape}')

# Show class distribution
specialty_counts = df['medical_specialty'].value_counts()
print(f'\nNumber of specialties: {len(specialty_counts)}')

plt.figure(figsize=(14, 6))
specialty_counts.plot(kind='barh', color='steelblue')
plt.title('Medical Specialty Distribution')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

## 5. Text Cleaning & Label Encoding

In [ ]:
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = ' '.join([w for w in text.split() if w not in STOPWORDS])
    return text

df['clean_text'] = df['transcription'].apply(clean_text)

texts = df['clean_text'].tolist()
labels_text = df['medical_specialty'].tolist()

le = LabelEncoder()
labels = le.fit_transform(labels_text)
num_classes = len(le.classes_)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f'Training samples : {len(train_texts)}')
print(f'Validation samples: {len(val_texts)}')
print(f'Number of classes : {num_classes}')

---
# Part A: LSTM Baseline Model (TensorFlow/Keras)

## 6. LSTM — Tokenization & Padding

In [ ]:
MAX_WORDS = 10000
MAX_SEQ_LEN = 128

lstm_tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
lstm_tokenizer.fit_on_texts(train_texts)

X_train_lstm = pad_sequences(lstm_tokenizer.texts_to_sequences(train_texts),
                             maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
X_val_lstm   = pad_sequences(lstm_tokenizer.texts_to_sequences(val_texts),
                             maxlen=MAX_SEQ_LEN, padding='post', truncating='post')

print(f'X_train shape: {X_train_lstm.shape}')
print(f'X_val   shape: {X_val_lstm.shape}')

## 7. LSTM — Build & Train

In [ ]:
def build_lstm_model(vocab_size=MAX_WORDS, embedding_dim=128,
                     max_seq_length=MAX_SEQ_LEN, num_classes=num_classes):
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_seq_length),
        SpatialDropout1D(0.2),
        LSTM(128, return_sequences=False),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam', metrics=['accuracy'])
    return model

lstm_model = build_lstm_model()
lstm_model.summary()

In [ ]:
OUTPUT_DIR = '/kaggle/working/' if os.path.exists('/kaggle/working') else './'

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'lstm_model.h5'), save_best_only=True
    )
]

history = lstm_model.fit(
    X_train_lstm, np.array(train_labels),
    epochs=10, batch_size=32,
    validation_data=(X_val_lstm, np.array(val_labels)),
    callbacks=callbacks
)

## 8. LSTM — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('LSTM Loss'); axes[0].legend()
axes[1].plot(history.history['accuracy'], label='Train Acc')
axes[1].plot(history.history['val_accuracy'], label='Val Acc')
axes[1].set_title('LSTM Accuracy'); axes[1].legend()
plt.tight_layout(); plt.show()

## 9. LSTM — Evaluation

In [ ]:
lstm_preds = np.argmax(lstm_model.predict(X_val_lstm, verbose=0), axis=1)
lstm_acc = accuracy_score(val_labels, lstm_preds)
print(f'LSTM Validation Accuracy: {lstm_acc:.4f}\n')

unique = np.unique(np.concatenate((val_labels, lstm_preds)))
names  = [le.classes_[i] for i in unique]
print(classification_report(val_labels, lstm_preds,
                            labels=unique, target_names=names))

## 10. LSTM — Save Artefacts

In [ ]:
# Save tokenizer + class labels
with open(os.path.join(OUTPUT_DIR, 'lstm_tokenizer.pkl'), 'wb') as f:
    pickle.dump(lstm_tokenizer, f, protocol=pickle.HIGHEST_PROTOCOL)
np.save(os.path.join(OUTPUT_DIR, 'classes.npy'), le.classes_)

print('Saved: lstm_model.h5, lstm_tokenizer.pkl, classes.npy')
print('Download these from the Output tab to use in the Flask app.')

---
# Part B: BioBERT Fine-Tuned Model (PyTorch)

## 11. BioBERT — Tokenization

In [ ]:
MODEL_NAME = 'dmis-lab/biobert-v1.1'

print(f'Loading tokenizer: {MODEL_NAME}')
bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print('Tokenizing training set...')
train_enc = bert_tokenizer(train_texts, truncation=True,
                           padding=True, max_length=128)
print('Tokenizing validation set...')
val_enc   = bert_tokenizer(val_texts, truncation=True,
                           padding=True, max_length=128)
print('Done.')

## 12. BioBERT — DataLoaders

In [ ]:
class MedicalDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MedicalDataset(train_enc, train_labels)
val_dataset   = MedicalDataset(val_enc, val_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## 13. BioBERT — Load Model

In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_classes
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model.to(device)
print(f'Model on: {device}')

## 14. BioBERT — Training Loop

In [ ]:
optimizer = AdamW(bert_model.parameters(), lr=2e-5)
loss_fn = CrossEntropyLoss()
EPOCHS = 3

train_losses = []
val_accs = []

for epoch in range(EPOCHS):
    bert_model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch in pbar:
        optimizer.zero_grad()
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs = batch['labels'].to(device)

        out = bert_model(ids, attention_mask=mask)
        loss = loss_fn(out.logits, labs)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    # Quick validation
    bert_model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in val_loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labs = batch['labels'].to(device)
            preds = torch.argmax(bert_model(ids, attention_mask=mask).logits, dim=1)
            correct += (preds == labs).sum().item()
            total += labs.size(0)
    val_acc = correct / total
    val_accs.append(val_acc)
    print(f'  Avg Loss: {avg_loss:.4f}  |  Val Accuracy: {val_acc:.4f}')

## 15. BioBERT — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(1, EPOCHS+1), train_losses, 'o-')
axes[0].set_title('BioBERT Training Loss'); axes[0].set_xlabel('Epoch')
axes[1].plot(range(1, EPOCHS+1), val_accs, 'o-', color='green')
axes[1].set_title('BioBERT Validation Accuracy'); axes[1].set_xlabel('Epoch')
plt.tight_layout(); plt.show()

## 16. BioBERT — Full Evaluation

In [ ]:
bert_model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for batch in tqdm(val_loader, desc='Evaluating'):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs = batch['labels'].to(device)
        preds = torch.argmax(bert_model(ids, attention_mask=mask).logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labs.cpu().numpy())

bert_acc = accuracy_score(all_true, all_preds)
print(f'BioBERT Validation Accuracy: {bert_acc:.4f}\n')

unique = np.unique(np.concatenate((all_true, all_preds)))
names  = [le.classes_[i] for i in unique]
print(classification_report(all_true, all_preds,
                            labels=unique, target_names=names))

## 17. Confusion Matrix (BioBERT)

In [ ]:
cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('BioBERT Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=90); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

## 18. Save BioBERT Model

In [ ]:
bert_save_dir = os.path.join(OUTPUT_DIR, 'biobert_finetuned')
os.makedirs(bert_save_dir, exist_ok=True)
bert_model.save_pretrained(bert_save_dir)
bert_tokenizer.save_pretrained(bert_save_dir)
print(f'BioBERT saved to: {bert_save_dir}')

## 19. Model Comparison

In [ ]:
print('=' * 50)
print(f'  LSTM Accuracy   : {lstm_acc:.4f}')
print(f'  BioBERT Accuracy: {bert_acc:.4f}')
print('=' * 50)
print()
winner = 'BioBERT' if bert_acc > lstm_acc else 'LSTM'
print(f'Winner: {winner}')

models = ['LSTM', 'BioBERT']
accs   = [lstm_acc, bert_acc]
colors = ['#4fc3f7', '#7c4dff']
plt.figure(figsize=(8, 5))
bars = plt.bar(models, accs, color=colors, width=0.5)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc:.2%}', ha='center', fontsize=14, fontweight='bold')
plt.ylim(0, max(accs) + 0.1)
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.tight_layout(); plt.show()

## 20. Sample Predictions

In [ ]:
test_samples = [
    'Patient presents with severe chest pain radiating to left arm and shortness of breath',
    'Child has persistent cough and high fever for three days with runny nose',
    'Fractured right femur after a fall, swelling and inability to bear weight',
    'Chronic lower back pain with numbness in both legs and difficulty walking',
    'Blurry vision in left eye and recurrent headaches with nausea'
]

cleaned = [clean_text(s) for s in test_samples]

# LSTM predictions
seq = pad_sequences(lstm_tokenizer.texts_to_sequences(cleaned),
                    maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
lstm_p = np.argmax(lstm_model.predict(seq, verbose=0), axis=1)

# BioBERT predictions
enc = bert_tokenizer(cleaned, truncation=True, padding=True,
                     max_length=128, return_tensors='pt')
with torch.no_grad():
    bert_out = bert_model(enc['input_ids'].to(device),
                         attention_mask=enc['attention_mask'].to(device))
bert_p = torch.argmax(bert_out.logits, dim=1).cpu().numpy()

print(f'{"Symptom":<85} {"LSTM":<30} {"BioBERT"}')
print('-' * 145)
for s, lp, bp in zip(test_samples, lstm_p, bert_p):
    print(f'{s[:82]:<85} {le.classes_[lp]:<30} {le.classes_[bp]}')